# M04-P02 — Multimodal Incident Analyzer

En M01-P02 construimos `Analyze` para convertir **texto libre** en una salida estructurada y validada.

Ahora ampliamos la misma capacidad para trabajar con:

```text
texto + imagen
```

Caso de uso:

```text
descripción de incidencia
+
captura de dashboard / error / activo
        ↓
modelo multimodal
        ↓
análisis estructurado
```

La práctica utiliza un modelo multimodal real de Amazon Bedrock mediante Converse.

## 1. Qué queremos demostrar

No queremos una demo aislada de reconocimiento de imágenes. Queremos comprobar qué aporta una evidencia visual a una aplicación empresarial que ya existe.

Durante la práctica:

1. generaremos una captura sintética de un dashboard;
2. comprobaremos qué observa el modelo;
3. compararemos **texto solo** frente a **texto + imagen**;
4. separaremos hechos, evidencia visual, hipótesis e incertidumbres;
5. provocaremos una contradicción entre texto e imagen;
6. integraremos la nueva capacidad en `Enterprise GenAI Assistant`.

In [ ]:
from pathlib import Path
import json
import time

import boto3
import matplotlib.pyplot as plt
import pandas as pd

from enterprise_genai.config import get_settings
from enterprise_genai.analysis import (
    analyze_incident,
    analyze_incident_multimodal,
)

settings = get_settings()
session = boto3.Session(
    profile_name=settings.aws_profile or None,
    region_name=settings.aws_region,
)
runtime = session.client("bedrock-runtime")
identity = session.client("sts").get_caller_identity()
MODEL_ID = settings.bedrock_text_model_id

print("AWS identity:", identity["Arn"])
print("Region:", settings.aws_region)
print("Model:", MODEL_ID)

## 2. Crear una evidencia visual controlada

Para que todos trabajemos con la misma evidencia generamos un dashboard sintético.

La captura añade información que **no aparece explícitamente** en el texto de la incidencia:

- crecimiento de errores de autenticación;
- `VPN AUTH BACKEND = DEGRADED`;
- `CORPORATE WEB = OK`;
- `VPN GATEWAY = OK`.

Definimos una función para poder reutilizar el mismo dashboard más adelante con una contradicción deliberada.

In [ ]:
OUT_DIR = Path("outputs/m04_p02")
OUT_DIR.mkdir(parents=True, exist_ok=True)

times = ["07:50","08:00","08:05","08:10","08:15","08:20","08:25","08:30"]
auth_errors = [1, 1, 2, 2, 12, 31, 46, 48]

def build_dashboard(path, corporate_web_status="OK", corporate_web_detail="No active alerts"):
    fig = plt.figure(figsize=(12, 6))
    gs = fig.add_gridspec(2, 3, height_ratios=[2.2, 1])

    ax = fig.add_subplot(gs[0, :])
    ax.plot(times, auth_errors, marker="o", linewidth=2)
    ax.axvline("08:15", linestyle="--", alpha=0.7)
    ax.set_title("ACME — Remote Access Operations Dashboard")
    ax.set_ylabel("VPN authentication failures / min")
    ax.set_xlabel("Time")
    ax.grid(alpha=0.25)

    cards = [
        ("VPN AUTH BACKEND", "DEGRADED", "Intermittent authentication errors"),
        ("CORPORATE WEB", corporate_web_status, corporate_web_detail),
        ("VPN GATEWAY", "OK", "Traffic within expected range"),
    ]

    for i, (title, status, detail) in enumerate(cards):
        a = fig.add_subplot(gs[1, i])
        a.axis("off")
        a.text(0.5, 0.78, title, ha="center", va="center", fontsize=12, weight="bold")
        a.text(0.5, 0.46, status, ha="center", va="center", fontsize=18, weight="bold")
        a.text(0.5, 0.18, detail, ha="center", va="center", fontsize=9)

    plt.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

DASHBOARD_PATH = OUT_DIR / "vpn_auth_dashboard.png"
build_dashboard(DASHBOARD_PATH)

print("Imagen:", DASHBOARD_PATH)
print("Tamaño:", DASHBOARD_PATH.stat().st_size, "bytes")

## 3. Abrir primero la caja: petición multimodal sin estructura

Antes de pedir JSON estructurado comprobamos que el modelo puede inspeccionar la imagen.

La petición contiene dos bloques:

```text
content
├── text
└── image
```

Con `boto3`, los bytes de la imagen se envían directamente.

In [ ]:
image_bytes = DASHBOARD_PATH.read_bytes()

started = time.perf_counter()
raw_response = runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "Analiza únicamente lo que sea directamente visible en esta captura. "
                        "Enumera evidencias observables y evita formular una causa raíz. "
                        "Si una cifra no puede leerse con claridad, no la inventes."
                    )
                },
                {
                    "image": {
                        "format": "png",
                        "source": {"bytes": image_bytes},
                    }
                },
            ],
        }
    ],
    inferenceConfig={"temperature": 0.0, "maxTokens": 500},
)
client_ms = (time.perf_counter() - started) * 1000

raw_text = raw_response["output"]["message"]["content"][0]["text"]
print(raw_text)
print()
print("Stop reason:", raw_response.get("stopReason"))
print("Usage:", raw_response.get("usage", {}))
print("Bedrock metrics:", raw_response.get("metrics", {}))
print(f"Client time: {client_ms:.0f} ms")

### Qué debemos observar

El modelo debería extraer evidencias como el aumento de fallos de autenticación, el backend degradado y los otros componentes en estado `OK`.

No debería convertir automáticamente esas observaciones en una causa raíz confirmada.

> Una lectura visual de una gráfica puede ser aproximada. Para obtener cifras exactas, una aplicación debería consultar la fuente de métricas, no depender de una captura.

## 4. Texto solo: punto de partida de M01-P02

Usaremos una incidencia deliberadamente incompleta:

> Desde las 08:15, 73 usuarios de ACME no pueden autenticarse en la VPN.  
> El servicio web corporativo funciona correctamente.  
> El equipo de operaciones está investigando.

El texto **no dice** que el backend de autenticación esté degradado.

In [ ]:
INCIDENT = (
    "Desde las 08:15, 73 usuarios de ACME no pueden autenticarse en la VPN. "
    "El servicio web corporativo funciona correctamente. "
    "El equipo de operaciones está investigando."
)

text_only = analyze_incident(INCIDENT)
text_analysis = text_only["analysis"]

print(json.dumps(text_analysis.model_dump(), indent=2, ensure_ascii=False))
print()
print("Stop reason:", text_only["stop_reason"])
print("Usage:", text_only["usage"])

## 5. Texto + imagen: análisis multimodal estructurado

Ahora enviamos exactamente el mismo texto junto con la captura.

El schema multimodal añade:

```text
visual_summary
visual_evidence
visual_uncertainties
text_image_consistency
```

La regla es:

```text
observed_facts
→ hechos sostenibles con el conjunto de evidencias

visual_evidence
→ información directamente observable en la imagen

hypotheses
→ explicaciones posibles, no hechos confirmados
```

In [ ]:
multi = analyze_incident_multimodal(
    INCIDENT,
    image_bytes=image_bytes,
    image_format="png",
)
multi_analysis = multi["analysis"]

print(json.dumps(multi_analysis.model_dump(), indent=2, ensure_ascii=False))
print()
print("Stop reason:", multi["stop_reason"])
print("Usage:", multi["usage"])
print("Bedrock metrics:", multi["metrics"])

## 6. Comparar qué aporta la imagen

La pregunta no es si la respuesta multimodal es más larga, sino si aparecen hechos útiles que no podían obtenerse del texto.

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "enfoque": "Texto",
            "categoria": text_analysis.category,
            "servicio": text_analysis.affected_service,
            "usuarios": text_analysis.affected_users,
            "n_hechos": len(text_analysis.observed_facts),
            "evidencia_visual": "No aplica",
            "consistencia": "No aplica",
            "tokens": text_only["usage"].get("totalTokens"),
        },
        {
            "enfoque": "Texto + imagen",
            "categoria": multi_analysis.category,
            "servicio": multi_analysis.affected_service,
            "usuarios": multi_analysis.affected_users,
            "n_hechos": len(multi_analysis.observed_facts),
            "evidencia_visual": len(multi_analysis.visual_evidence),
            "consistencia": multi_analysis.text_image_consistency,
            "tokens": multi["usage"].get("totalTokens"),
        },
    ]
)

display(comparison)

## 7. Inspeccionar solo la evidencia visual

Queremos hacer visible qué información proviene de la captura y qué parte es razonamiento del modelo.

In [ ]:
print("Resumen visual:")
print(multi_analysis.visual_summary)

print("\nEvidencias visuales:")
for item in multi_analysis.visual_evidence:
    print("-", item)

print("\nIncertidumbres visuales:")
if multi_analysis.visual_uncertainties:
    for item in multi_analysis.visual_uncertainties:
        print("-", item)
else:
    print("- Ninguna declarada")

print("\nConsistencia texto/imagen:", multi_analysis.text_image_consistency)

## 8. Hecho visual no significa causa raíz

Esta sección es deliberadamente conceptual; no requiere ejecutar código.

Si la captura muestra:

```text
VPN AUTH BACKEND = DEGRADED
```

podemos afirmar:

```text
el dashboard muestra el backend degradado
```

pero no:

```text
la causa raíz es X
```

El estado visual es una evidencia. La causa raíz sigue necesitando investigación, logs, métricas y otras fuentes.

## 9. Contradicción entre modalidades

Ahora sí hacemos un experimento ejecutable.

Mantendremos el texto original, que afirma:

```text
El servicio web corporativo funciona correctamente.
```

pero generaremos otra captura donde:

```text
CORPORATE WEB = DEGRADED
```

La aplicación no debería elegir silenciosamente una fuente: debería hacer visible la contradicción.

In [ ]:
CONTRADICTORY_PATH = OUT_DIR / "vpn_auth_dashboard_contradictory.png"

build_dashboard(
    CONTRADICTORY_PATH,
    corporate_web_status="DEGRADED",
    corporate_web_detail="Active service degradation",
)

contradictory_bytes = CONTRADICTORY_PATH.read_bytes()

contradiction = analyze_incident_multimodal(
    INCIDENT,
    image_bytes=contradictory_bytes,
    image_format="png",
)
contradiction_analysis = contradiction["analysis"]

print("Consistencia:", contradiction_analysis.text_image_consistency)
print("Revisión humana:", contradiction_analysis.human_review_required)
print("Motivo:", contradiction_analysis.review_reason)

print("\nEvidencias visuales:")
for item in contradiction_analysis.visual_evidence:
    print("-", item)

print("\nHechos observados:")
for item in contradiction_analysis.observed_facts:
    print("-", item)

print("\nStop reason:", contradiction["stop_reason"])
print("Usage:", contradiction["usage"])

### Qué esperamos observar

El resultado esperado es:

```text
text_image_consistency = inconsistent
```

o, como mínimo, `partially_consistent` con una explicación explícita de la contradicción.

También esperamos:

```text
human_review_required = true
```

La lección es que **fusionar fuentes no significa asumir que todas coinciden**.

## 10. Integración en Enterprise GenAI Assistant

La página `Analyze` acepta una imagen opcional.

Sin imagen:

```text
texto
→ Structured Incident Analyzer
```

Con imagen:

```text
texto + imagen
→ Multimodal Incident Analyzer
```

La interfaz muestra evidencia visual, incertidumbres, consistencia texto/imagen, métricas y JSON estructurado.

Arranca la aplicación:

```bash
streamlit run app/Home.py
```

y abre **Analyze**.

## 11. Qué debes llevarte

La evolución de la aplicación es:

```text
M01-P02
texto
→ análisis estructurado

M04-P02
texto + imagen
→ análisis multimodal estructurado
```

La imagen puede:

- confirmar el texto;
- aportar información nueva;
- contradecir otras fuentes;
- aumentar la necesidad de revisión humana.

Ésta es la utilidad real de incorporar visión a una aplicación empresarial.